# Tableau용 정제 데이터 생성

기획안_설계.md 5번(전처리 규칙)의 보정 로직을 Python(pandas)에서 미리 적용해 Tableau에 바로 연결할 수 있는 정제 CSV 3개를 만든다.

- `stores_정제.csv`: 프랜차이즈/일반/전체/개업 점포 수 보정 컬럼 추가 (02_EDA_상권쇠퇴분석.ipynb STEP1 로직 재사용)
- `master_정제.csv`: 좌표계 변환(EPSG:5181 → WGS84 위도/경도) 컬럼 추가
- `attraction_facilities_정제.csv`: 세부 시설 컬럼 결측치를 0으로 대체

Tableau는 이 정제 CSV들을 상권_코드 기준으로 관계(Relationship) 설정해서 연결하고, KPI 계산 필드는 Tableau 안에서 만든다(8-1 로드맵 3단계).

In [1]:
import pandas as pd
import numpy as np
from pyproj import Transformer

# 인코딩은 파일마다 다름 (데이터_명세.md 참고): stores/master는 cp949, attraction_facilities는 utf-8-sig
df_stores = pd.read_csv('../data/seoul_commercial_district_stores_2025.csv', encoding='cp949')
df_master = pd.read_csv('../data/seoul_commercial_district_master_2025.csv', encoding='cp949')
df_facilities = pd.read_csv('../data/seoul_commercial_district_attraction_facilities_2025.csv', encoding='utf-8-sig')

print('stores:', df_stores.shape)
print('master:', df_master.shape)
print('facilities:', df_facilities.shape)

stores: (304775, 12)
master: (1650, 11)
facilities: (6312, 25)


## 1. stores 정제 — 점포 수 보정 (02번 노트북 STEP1과 동일 로직)

In [2]:
df_stores_정제 = df_stores.copy()

# 1) 프랜차이즈_점포_수 보정: 공식(전체=일반+프랜차이즈)이 깨진 행만 비율(%)->개수로 역산
전체_보정후_예상값 = (df_stores_정제['일반_점포_수'] * df_stores_정제['프랜차이즈_점포_수'] / 100).round(0).astype(int)
원래_공식_성립 = df_stores_정제['전체_점포_수'] == (df_stores_정제['일반_점포_수'] + df_stores_정제['프랜차이즈_점포_수'])
df_stores_정제['프랜차이즈_점포_수_보정'] = np.where(원래_공식_성립, df_stores_정제['프랜차이즈_점포_수'], 전체_보정후_예상값)

# 2) 일반_점포_수 보정: 전체 - 프랜차이즈(보정), 음수는 0으로 클리핑
df_stores_정제['일반_점포_수_보정'] = (df_stores_정제['전체_점포_수'] - df_stores_정제['프랜차이즈_점포_수_보정']).clip(lower=0)

# 3) 전체_점포_수 보정: 원본과 (일반+프랜차이즈 보정) 중 더 큰 값
전체_후보 = df_stores_정제['일반_점포_수_보정'] + df_stores_정제['프랜차이즈_점포_수_보정']
df_stores_정제['전체_점포_수_보정'] = np.maximum(df_stores_정제['전체_점포_수'], 전체_후보)

# 4) 개업_점포_수 클리핑: 공식 보정은 적용하지 않고, 전체_점포_수_보정을 초과하는 값만 눌러줌
#    (05_개업점포수_보정_검증.ipynb에서 공식 보정의 위험성을 확인하고 클리핑만 적용하기로 결정)
df_stores_정제['개업_점포_수_보정'] = np.minimum(df_stores_정제['개업_점포_수'], df_stores_정제['전체_점포_수_보정'])

# 검증: 보정 후 공식이 100% 성립하는지, 음수/초과 행이 없는지 확인
mismatch = (df_stores_정제['전체_점포_수_보정'] != (df_stores_정제['일반_점포_수_보정'] + df_stores_정제['프랜차이즈_점포_수_보정'])).sum()
개업_초과 = (df_stores_정제['개업_점포_수_보정'] > df_stores_정제['전체_점포_수_보정']).sum()
print(f'보정 후 공식 불일치 행: {mismatch} (0이어야 함)')
print(f'개업_점포_수_보정 > 전체_점포_수_보정 인 행: {개업_초과} (0이어야 함)')

보정 후 공식 불일치 행: 0 (0이어야 함)
개업_점포_수_보정 > 전체_점포_수_보정 인 행: 0 (0이어야 함)


In [3]:
df_stores_정제.to_csv('../data_clean/stores_clean.csv', index=False, encoding='utf-8-sig')
print('stores_clean.csv 저장 완료:', df_stores_정제.shape)
df_stores_정제.head()

stores_clean.csv 저장 완료: (304775, 16)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수,프랜차이즈_점포_수_보정,일반_점포_수_보정,전체_점포_수_보정,개업_점포_수_보정
0,20251,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,13,0,8,1,0,12,12,8
1,20251,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,0,1,1,0
2,20251,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,2,2,0,0,0,0,2,2,0
3,20251,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,33,1,0,2,2,2
4,20251,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,2,2,0,0,0,0,2,2,0


## 2. master 정제 — 좌표계 변환 (EPSG:5181 → WGS84)

엑스좌표_값/와이좌표_값은 EPSG:5181(한국 중부원점) 기준. Tableau 지도 기능은 위도/경도(WGS84)가 필요하므로 pyproj로 변환.

In [4]:
df_master_정제 = df_master.copy()

# always_xy=True: 입력/출력 순서를 (x, y) = (경도, 위도) 순서로 고정 (좌표계 변환 시 축 순서 혼동 방지)
transformer = Transformer.from_crs('EPSG:5181', 'EPSG:4326', always_xy=True)
경도, 위도 = transformer.transform(df_master_정제['엑스좌표_값'].values, df_master_정제['와이좌표_값'].values)

df_master_정제['경도'] = 경도
df_master_정제['위도'] = 위도

# 검증: 서울시 범위(대략 위도 37.4~37.7, 경도 126.7~127.2) 안에 들어오는지 확인
print('위도 범위:', df_master_정제['위도'].min(), '~', df_master_정제['위도'].max())
print('경도 범위:', df_master_정제['경도'].min(), '~', df_master_정제['경도'].max())

위도 범위: 37.43459211622573 ~ 37.68982127455393
경도 범위: 126.80201817298321 ~ 127.17376605352426


In [5]:
df_master_정제.to_csv('../data_clean/master_clean.csv', index=False, encoding='utf-8-sig')
print('master_clean.csv 저장 완료:', df_master_정제.shape)
df_master_정제[['상권_코드_명', '엑스좌표_값', '와이좌표_값', '경도', '위도']].head()

master_clean.csv 저장 완료: (1650, 13)


,상권_코드_명,엑스좌표_값,와이좌표_값,경도,위도
0,황학동벼룩시장,201642,452260,127.018587,37.569879
1,배화여자대학교(박노수미술관),197093,453418,126.967090,37.580309
2,자하문터널,196991,455057,126.965928,37.595076
3,평창동서측,197064,456643,126.966748,37.609366
4,정독도서관,198581,453781,126.983935,37.583584


## 3. attraction_facilities 정제 — 결측치 0 처리

세부 시설 컬럼의 빈 값은 "해당 시설 없음"으로 해석해 0으로 대체 (기획안_설계.md 5번 규칙).

In [6]:
df_facilities_정제 = df_facilities.copy()

# 시설 수 컬럼만 선택 (코드/이름 컬럼 제외) — '_수'로 끝나는 컬럼
시설_컬럼 = [c for c in df_facilities_정제.columns if c.endswith('_수')]
df_facilities_정제[시설_컬럼] = df_facilities_정제[시설_컬럼].fillna(0)

print('결측치 처리 후 남은 결측치 개수:', df_facilities_정제[시설_컬럼].isna().sum().sum(), '(0이어야 함)')

결측치 처리 후 남은 결측치 개수: 0 (0이어야 함)


In [7]:
df_facilities_정제.to_csv('../data_clean/attraction_facilities_clean.csv', index=False, encoding='utf-8-sig')
print('attraction_facilities_clean.csv 저장 완료:', df_facilities_정제.shape)
df_facilities_정제.head()

attraction_facilities_clean.csv 저장 완료: (6312, 25)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,집객시설_수,관공서_수,은행_수,종합병원_수,일반_병원_수,...,대학교_수,백화점_수,슈퍼마켓_수,극장_수,숙박_시설_수,공항_수,철도_역_수,버스_터미널_수,지하철_역_수,버스_정거장_수
0,20254,U,관광특구,3001496,강남 마이스 관광특구,35,1.0,8.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,8.0
1,20254,U,관광특구,3001495,잠실 관광특구,158,3.0,13.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,20.0
2,20254,U,관광특구,3001494,종로?청계 관광특구,246,5.0,17.0,0.0,0.0,...,0.0,0.0,5.0,2.0,15.0,0.0,0.0,0.0,3.0,28.0
3,20254,U,관광특구,3001493,동대문패션타운 관광특구,144,3.0,12.0,1.0,1.0,...,0.0,0.0,3.0,2.0,1.0,0.0,0.0,0.0,5.0,17.0
4,20254,U,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,594,14.0,57.0,0.0,0.0,...,1.0,1.0,2.0,5.0,9.0,0.0,0.0,0.0,3.0,44.0


## 결론

`data/` 폴더에 3개 정제 CSV가 생성되었다:
- `stores_정제.csv` — 전체_점포_수_보정, 일반_점포_수_보정, 프랜차이즈_점포_수_보정, 개업_점포_수_보정 컬럼 추가 (원본 컬럼은 그대로 유지)
- `master_정제.csv` — 위도, 경도 컬럼 추가
- `attraction_facilities_정제.csv` — 시설 수 컬럼 결측치 0 처리

다음 단계(13-3): 이 3개 CSV를 Tableau에 연결하고 상권_코드 기준으로 관계(Relationship) 설정.

## 4. attraction_facilities 정제 — long 형태로 피벗 (Tableau Dimension#2용)

화면2 Section1 Dimension#2(집객시설 특성)를 만들려면 22개 시설 컬럼(은행_수, 약국_수 등)을
Tableau에서 각각 계산 필드로 반복하는 대신, `시설_종류`라는 차원 하나로 묶어서 다루는 게 훨씬 간단하다
(03번 노트북에서 서비스_업종_코드_명을 차원으로 쓴 것과 같은 구조).

`pd.melt`로 wide(컬럼별) → long(행별) 형태로 변환한다.

In [8]:
# 집객시설은 연 1회(4분기) 갱신이라 1~3분기 값이 4분기와 동일하게 복제되어 있음
# (04번 노트북 STEP2에서 확인한 사실) -> 20254 한 분기만 남기고 melt해야 중복이 안 생김
df_facilities_20254 = df_facilities_정제[df_facilities_정제["기준_년분기_코드"] == 20254].copy()

# wide -> long 변환: 시설 컬럼 22개를 "시설_종류" 하나의 차원으로 펼침
id_cols = ["기준_년분기_코드", "상권_구분_코드", "상권_구분_코드_명", "상권_코드", "상권_코드_명"]
시설_컬럼 = [c for c in df_facilities_정제.columns if c.endswith("_수") and c != "집객시설_수"]

df_facilities_long = df_facilities_20254.melt(
    id_vars=id_cols,
    value_vars=시설_컬럼,
    var_name="시설_종류",
    value_name="시설_수",
)

print("long 형태 shape:", df_facilities_long.shape, f"({len(df_facilities_20254)}행 x {len(시설_컬럼)}개 시설 = {len(df_facilities_20254)*len(시설_컬럼)}행이어야 함)")
df_facilities_long.head()

long 형태 shape: (29982, 7) (1578행 x 19개 시설 = 29982행이어야 함)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,시설_종류,시설_수
0,20254,U,관광특구,3001496,강남 마이스 관광특구,관공서_수,1.0
1,20254,U,관광특구,3001495,잠실 관광특구,관공서_수,3.0
2,20254,U,관광특구,3001494,종로?청계 관광특구,관공서_수,5.0
3,20254,U,관광특구,3001493,동대문패션타운 관광특구,관공서_수,3.0
4,20254,U,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,관공서_수,14.0


In [9]:
df_facilities_long.to_csv('../data_clean/attraction_facilities_long.csv', index=False, encoding='utf-8-sig')
print('attraction_facilities_long.csv 저장 완료:', df_facilities_long.shape)

attraction_facilities_long.csv 저장 완료: (29982, 7)


## 5. 위험상권_여부를 Python에서 미리 계산해 CSV에 포함 (Tableau 관계 fan-out 방지)

Tableau에서 `위험상권_여부`를 stores_clean 기준 LOD(`{FIXED 상권_코드:...}`)로 만들어서
attraction_facilities_long 쪽 뷰(시설_종류가 행에 있는)에서 참조했더니, 관계(Relationship)가
stores_clean의 세밀한 그레인(상권_코드×기준_년분기_코드×서비스_업종_코드)까지 끌고 들어와서
집객시설 평균 계산에 같은 값이 업종 개수만큼 중복 카운트되는 문제가 발생했다
(약국_수 하나의 평균 계산에 들어가야 할 행이 299개여야 하는데 14,325개로 부풀려짐).

해결: 위험상권_여부를 Tableau LOD가 아니라 **Python에서 미리 계산**해서 상권_코드 단위로
확정한 뒤, stores_clean과 attraction_facilities_long 양쪽 CSV에 똑같이 컬럼으로 박아넣는다.
이러면 Tableau에서는 이 컬럼을 그냥 그대로 쓰기만 하면 되므로 fan-out이 원천적으로 발생하지 않는다.

(02번 노트북 STEP1·2·7과 동일한 로직으로 위험 상권을 재현 — 절대 기준 통과 299개에서
최소 표본 기준(전체_점포_수_보정 ≥ 30, STEP7)까지 적용한 최종 285개를 사용해야
기획안_설계.md 4-1번에 확정된 자치구 순위·비율과 일치한다)

In [10]:
# --- 위험 상권 판정 재현 (02번 노트북 STEP1·2·7과 동일 로직) ---
분모_안전 = df_stores_정제.groupby(["상권_코드", "기준_년분기_코드"])["전체_점포_수_보정"].sum()

집계 = df_stores_정제.groupby(["상권_코드", "기준_년분기_코드"], as_index=False).agg(
    전체=("전체_점포_수_보정", "sum"),
    개업=("개업_점포_수_보정", "sum"),
    폐업=("폐업_점포_수", "sum"),
)
집계["KPI1_순증감율"] = (집계["개업"] - 집계["폐업"]) / 집계["전체"]
집계["폐업률"] = 집계["폐업"] / 집계["전체"]

집계 = 집계.sort_values(["상권_코드", "기준_년분기_코드"]).reset_index(drop=True)
집계["폐업률_전분기"] = 집계.groupby("상권_코드")["폐업률"].shift(1)
집계["KPI2_폐업률증감"] = 집계["폐업률"] - 집계["폐업률_전분기"]

최신분기 = 집계[집계["기준_년분기_코드"] == 20254].copy()
위험_조건 = (최신분기["KPI1_순증감율"] < 0) & (최신분기["KPI2_폐업률증감"] > 0)
위험상권_코드_집합_절대기준 = set(최신분기.loc[위험_조건, "상권_코드"])

print(f"절대 기준 통과 위험 상권 수: {len(위험상권_코드_집합_절대기준)} (299여야 함)")

# --- 최소 표본 기준(STEP7): 전체_점포_수_보정 < 30인 상권은 위험 판정에서 제외 ---
# 비율 기반 위험 판정은 표본이 작을수록 노이즈에 취약하므로(기획안_설계.md 5번),
# 최신분기 기준 전체_점포_수_보정 합계가 30 미만인 상권을 위험상권 목록에서 뺀다.
소규모_상권_코드 = set(최신분기.loc[최신분기["전체"] < 30, "상권_코드"])
위험상권_코드_집합 = 위험상권_코드_집합_절대기준 - 소규모_상권_코드

print(f"최소 표본 필터 후 최종 위험 상권 수: {len(위험상권_코드_집합)} (285여야 함)")

절대 기준 통과 위험 상권 수: 299 (299여야 함)
최소 표본 필터 후 최종 위험 상권 수: 285 (285여야 함)


In [11]:
# stores_clean, attraction_facilities_long 양쪽에 위험상권_여부 컬럼 추가 후 재저장
df_stores_정제["위험상권_여부"] = df_stores_정제["상권_코드"].isin(위험상권_코드_집합)
df_facilities_long["위험상권_여부"] = df_facilities_long["상권_코드"].isin(위험상권_코드_집합)

df_stores_정제.to_csv('../data_clean/stores_clean.csv', index=False, encoding='utf-8-sig')
df_facilities_long.to_csv('../data_clean/attraction_facilities_long.csv', index=False, encoding='utf-8-sig')

print("재저장 완료")
print("stores_clean.csv 위험상권_여부=True 고유 상권 수:", df_stores_정제.loc[df_stores_정제["위험상권_여부"], "상권_코드"].nunique(), "(285여야 함)")
print("attraction_facilities_long.csv 위험상권_여부=True 고유 상권 수:", df_facilities_long.loc[df_facilities_long["위험상권_여부"], "상권_코드"].nunique(), "(285 이하, 시설 데이터에 없는 상권은 제외되므로)")

재저장 완료
stores_clean.csv 위험상권_여부=True 고유 상권 수: 285 (285여야 함)
attraction_facilities_long.csv 위험상권_여부=True 고유 상권 수: 283 (285 이하, 시설 데이터에 없는 상권은 제외되므로)
